In [27]:
import pandas as pd
import numpy as np
import requests
import pickle
from collections import deque
from functools import reduce
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 150)
from warnings import simplefilter
simplefilter(action="ignore", category=pd.errors.PerformanceWarning)
import time
from collections import deque
import geopandas as gpd

In [35]:
df = pd.read_csv(r'G:\DATA\REQUESTS_CLIENTS\GNRC\20260729_Datacenters/FracTracker0929.csv')

In [42]:
df['id'] = range(len(df))
df['summer'] = 1
aside_cols = ['id', 'address', 'information_source', 'info_source_1', 'info_source_2', 'info_source_3', 'info_source_4', 'info_source_5', 'info_source_6', 'info_source_7', 
           'info_source_8', 'date_created', 'date_updated', 'community_group_website_1', 'community_group_website_2', 'petition_url', 'other_info', 'address', 'tenant', 'mw']
aside_df = df[aside_cols]
df_filtered = df.drop(columns=aside_cols, errors = 'ignore')

In [43]:
df_filtered.head(2)

,facility_name,city,state,zip,county,lat,long,status,location_confidence,purpose,operator_name,sizerank,power_source,dedicated_power_plant,number_of_generators,number_of_buildings,cooling_source,cooling_type,facility_size_sqft,property_size_acres,project_cost,expected_date_online,community_pushback,advocacy_information,resistance_status,nda,summer
0,Stak Energy Data Center,North Slope Borough,AK,99734.0,North Slope,69.90071,-148.81477,Proposed,Medium,AI/Cloud Computing,Stak,"Mega campus (>1,000 MW)",Natural gas,Yes,NaN,NaN,Air,NaN,NaN,715.0,$500 million,2028,Unknown,NaN,Emerging Concern,NaN,1
1,Prudhoe Bay Data Center,Prudhoe Bay,AK,99734.0,North Slope,70.18478,-148.44000,Operating,Medium,NaN,"Far North Digital, LLC",Hyperscale (100-999 MW),NaN,NaN,NaN,NaN,Air,NaN,NaN,100.0,NaN,NaN,Unknown,NaN,NaN,NaN,1


In [44]:
df_filtered['purpose'].value_counts()

purpose
AI                                                                                            53
Crypto                                                                                         5
Bitcoin                                                                                        3
AI/Cloud Computing                                                                             2
AI/Cloud computing                                                                             2
Hyperscale Data Center Campus                                                                  2
AI-specific chips, software, high-performance\ncomputing                                       1
AI/cloud computing, digital communications                                                     1
Telecom / network data center                                                                  1
AI Data center and solar fam                                                                   1
AI/cloud-computing    

In [38]:
#block looking at different topical fields

# df_filtered['resistance_status'].unique()
#if it's null, it's unknown to us at least so let's just fill that in
df_filtered['resistance_status'] = df_filtered['resistance_status'].fillna('Unknown')

# df_filtered['sizerank'].unique()
#make names uniform for size categories
df_filtered['sizerank'] = df_filtered['sizerank'].replace({'Mega Campus (>1,000 MW)': 'Mega campus (>1,000 MW)'})

#location confidence - there are only 49 lows - I'm going to make the decision to DROP THE LOW CONFIDENCE BUT LEAVE THE MEDIUM
# #df_filtered['location_confidence'].value_counts()
df_filtered['location_confidence'] = df_filtered['location_confidence'].replace({'high': 'High'})
df_filtered = df_filtered.loc[df['location_confidence'] != 'Low'].drop(columns = 'location_confidence')

#Yes or No then Description
# boop = df['dedicated_power_plant'].str.split(pat = ' - ', expand = True)
df_filtered['dedicated_power_plant'] = boop[0]
df_filtered['dedicated_power_plant_desc'] = boop[1]

In [ ]:
#block looking at location descriptors

#state
#df['state'].unique()
df_filtered['state'] = df_filtered['state'].replace({'CO ': 'CO'})
#connect state fips
state_fips = { 'AK': '02', 'AL': '01', 'AR': '05', 'AZ': '04', 'CA': '06', 'CO': '08', 'CT': '09', 'DE': '10', 'FL': '12', 
              'GA': '13', 'HI': '15', 'IA': '19', 'ID': '16', 'IL': '17', 'IN': '18', 'KS': '20', 'KY': '21', 'LA': '22', 
              'MA': '25', 'MD': '24', 'ME': '23', 'MI': '26', 'MN': '27', 'MO': '29', 'MS': '28', 'MT': '30', 'NC': '37', 
              'ND': '38', 'NE': '31', 'NH': '33', 'NJ': '34', 'NM': '35', 'NV': '32', 'NY': '36', 'OH': '39', 'OK': '40', 
              'OR': '41', 'PA': '42', 'SC': '45', 'SD': '46', 'TN': '47', 'TX': '48', 'UT': '49', 'VA': '51', 'WA': '53', 
              'WI': '55', 'WV': '54', 'WY': '56'}
df_filtered['ST_FIPS'] = df_filtered['state'].map(state_fips)

# # addresses are very non-uniform - gonna have to use the user-created lat/lon and if address is ever really needed we can just revisit options

#only 11 null zip codes, might want to use this so go ahead and clean it up
#df_filtered['zip'].isna().sum()
df_filtered['zip'] = df_filtered['zip'].astype(str)
boop = df_filtered['zip'].str.split(pat = '.', expand = True)
df_filtered['zip'] = boop[0]

In [34]:
#block looking at geographic stuff

#great
print(df_filtered['lat'].isna().sum())
print(df_filtered['long'].isna().sum())

0
0
